In [2]:
# from tensorflow import keras
# model = keras.models.load_model("models/CNN_Manakov_full.keras") 
# model.save("models/CNN_Manakov_full.h5", save_format="h5")

In [3]:
# Run these lines only the first time you run this notebook

# !pip install tensorflow_addons gdown shap -q
# !pip install git+https://github.com/katarinagresova/DeepExperiment

# !wget https://github.com/ML-Bioinfo-CEITEC/miRBind/raw/main/Models/miRBind.h5
# !wget https://raw.githubusercontent.com/ML-Bioinfo-CEITEC/miRBind/graphs/Datasets/evaluation_set_1_1_CLASH2013_paper.tsv

In [4]:
import tensorflow

import json
import random
import cv2 
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import register_keras_serializable
# import tensorflow_addons as tfa
from IPython.display import Image, display
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from textwrap import wrap
import shap
from shap.plots.colors import red_transparent_blue
from Bio import SeqIO

from deepexperiment.alignment import Attrament
from deepexperiment.visualization import plot_alignment, plot_miRNA_importance, plotbar_miRNA_importance
from deepexperiment.utils import one_hot_encoding, one_hot_encoding_batch
from deepexperiment.interpret import DeepShap

from constants import HELA_TRANSFACTION_DATA, PSILAC_DATA, TARGETSCAN_COLUMN_TO_SEQUENCE

/home/jovyan/my-conda-envs/deepExperimentTF2.7/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Params

In [5]:
DATASET_NAME = 'refseq_id.mirna_fc'
# DATASET_NAME = 'gene_name.psilac'
# DATASET_NAME = 'refseq_id.HeLa_transfection'

# ID_COLUMN = "refseq_mrna"
ID_COLUMN = 'RefSeq ID'

SAVE_EXPL_SCORES = True

In [6]:
TS_mirnas = list(TARGETSCAN_COLUMN_TO_SEQUENCE.items())
# mirna_index = 5
# mirna_index = 0
# mirna_index = 1
# mirna_index = 2
# mirna_index = 3
# mirna_index = 4
mirna_index = 6
mirna_name = TS_mirnas[mirna_index][0]
my_miRNA = TS_mirnas[mirna_index][1]

mirna_sequences = [my_miRNA]
mirna_sequences, mirna_name

(['AGCAGCATTGTACAGGGCTATGA'], 'hsa-miR-103a-3p')

In [12]:
TS_mirnas

[('hsa-miR-16-5p', 'TAGCAGCACGTAAATATTGGCG'),
 ('hsa-miR-106b-5p', 'TAAAGTGCTGACAGTGCAGAT'),
 ('hsa-miR-200a-3p', 'TAACACTGTCTGGTAACGATGT'),
 ('hsa-miR-200b-3p', 'TAATACTGCCTGGTAATGATGA'),
 ('hsa-miR-215-5p', 'ATGACCTATGAATTGACAGAC'),
 ('hsa-let-7c-5p', 'TGAGGTAGTAGGTTGTATGGTT'),
 ('hsa-miR-103a-3p', 'AGCAGCATTGTACAGGGCTATGA')]

In [15]:
# SEQUENCE_SOURCE_PATH = "data/3utr/3utr.sequences.pkl"
# SEQUENCE_SOURCE_PATH = 'data/3utr/3utr.sequences.refseq_id.mirna_fc.pkl'
# SEQUENCE_SOURCE_PATH = f'data/3utr/3utr.sequences.{DATASET_NAME}.pkl'
# SEQUENCE_SOURCE_PATH = f'data/processed/GRCh37.p13 hg19/3utr.sequences.{DATASET_NAME}.pkl'

SEQUENCE_SOURCE_PATH = f'data/processed/GRCh37.p13 hg19/UCSC/3utr.sequences.{DATASET_NAME}.pkl'


sequence_source_df = pd.read_pickle(SEQUENCE_SOURCE_PATH)

In [14]:
sequence_source_df.columns

Index(['RefSeq ID', 'Gene symbol', 'hsa-miR-16-5p', 'hsa-miR-106b-5p',
       'hsa-miR-200a-3p', 'hsa-miR-200b-3p', 'hsa-miR-215-5p', 'hsa-let-7c-5p',
       'hsa-miR-103a-3p', 'knownGene.name', 'knownGene.chrom', 'kgAlias.kgID',
       'kgAlias.alias', 'kgXref.kgID', 'kgXref.mRNA', 'kgXref.geneSymbol',
       'kgXref.refseq', 'knownToEnsembl.name', 'knownToEnsembl.value',
       'knownToRefSeq.name', 'knownToRefSeq.value', 'ID_versioned', 'Name',
       'Description', 'sequence', 'Chromosome', 'Start', 'End', 'Strand',
       'Transcript ID', 'Representative transcript?', 'ensembl_id_no_version',
       'sequence_origin', 'utr3_length'],
      dtype='object')

In [11]:
# SAVE_EXPLAINABILITY_SCORES_PATH = "explainability_scores_{}.json".format(mirna_name)
# SAVE_EXPLAINABILITY_SCORES_PATH = "data/3utr.explainability_scores_{}.json".format(mirna_name)
# SAVE_EXPLAINABILITY_SCORES_PATH = f"data/3utr.sequences.refseq_id.mirna_fc.explainability_scores_{mirna_name}.json"
# SAVE_EXPLAINABILITY_SCORES_PATH = f"data/3utr.sequences.refseq_id.mirna_fc.explainability_scores_{mirna_name}.refseq_id.json"
# SAVE_EXPLAINABILITY_SCORES_PATH = f"debug/3utr.sequences.refseq_id.mirna_fc.explainability_scores_{mirna_name}.refseq_id.json"
# SAVE_EXPLAINABILITY_SCORES_PATH = f"data/3utr.sequences.{DATASET_NAME}.explainability_scores_{mirna_name}.json"

# SAVE_EXPLAINABILITY_SCORES_PATH = f"data/scanned/GRCh37.p13 hg19/UCSC/3utr.sequences.{DATASET_NAME}.explainability_scores_{mirna_name}.json"

# SAVE_EXPLAINABILITY_SCORES_PATH = f"data/scanned/GRCh37.p13 hg19/UCSC/3utr.sequences.{DATASET_NAME}.explainability_scores_{mirna_name}.json"

# SAVE_SCANNING_ERRORS_PATH = f"data/scanned/GRCh37.p13 hg19/UCSC/3utr.sequences.{DATASET_NAME}.explainability_scores_{mirna_name}.scanning_errors.txt"

In [9]:
# CHANGED naming of explainability_scores_ to gradient_scores_

SAVE_EXPLAINABILITY_SCORES_PATH = f"data/scanned/GRCh37.p13 hg19/UCSC/3utr.sequences.{DATASET_NAME}.gradient_scores_{mirna_name}.json"

SAVE_SCANNING_ERRORS_PATH = f"data/scanned/GRCh37.p13 hg19/UCSC/3utr.sequences.{DATASET_NAME}.gradient_scores_{mirna_name}.scanning_errors.txt"

In [10]:
PREDICTION_THRESHOLD = 0
random.seed(42)

### Load and preprocess the transcript data

In [11]:
sequence_source_df.shape[0], sequence_source_df[ID_COLUMN].nunique(), sequence_source_df[[ID_COLUMN]].dropna().shape[0]

(8288, 8288, 8288)

In [12]:
gene_symbol_to_seq = sequence_source_df[[ID_COLUMN, "sequence"]].set_index(ID_COLUMN).to_dict()['sequence']

### Get our deep learning model and miRNA data

In [13]:
"""# Loading model and the data"""

# model = keras.models.load_model("models/miRBind.h5")   # Old model from miRBind trained on Ago1 data
# model = keras.models.load_model("models/CNN_Manakov_full.keras") 
# model = keras.models.load_model("models/CNN_Manakov_full.h5") 
model = keras.models.load_model("models/CNN_Manakov_full.h5", compile=False)
# miRNA/miRNA/models/model_miRNA.h5
# model = keras.models.load_model("models/model_miRNA.h5")   # from Vasek's paper https://github.com/ML-Bioinfo-CEITEC/HybriDetector/blob/main/ML/Models/model_miRNA.h5
# model.summary()

2025-05-01 18:04:12.039455: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-01 18:04:12.600264: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 43556 MB memory:  -> device: 0, name: NVIDIA A40, pci bus id: 0000:21:00.0, compute capability: 8.6


In [14]:
samples = pd.read_csv('evaluation_set_1_1_CLASH2013_paper.tsv', sep='\t')
samples.head()

,miRNA,gene,label
0,TAGGTAGTTTCATGTTGTTG,AGACCAGCAACAAACTACCAACACTGTAGAGGAGCCCCTGGATCTT...,1
1,GCAGTCCATGGGCATATACA,CAGCCAAGCCCAGGCTCGCAGTATCAATAAAGCAGTTCTCTGAGGA...,1
2,CAAAGTGCTTACAGTGCAGG,TGAGAACACTTAAAGTTCAAATAGAAATCATTTCTGAAGACAAAAG...,1
3,GAGGGTTGGGTGGAGGCTCT,AATGCCCCAAAATCCAAACCTTCCTGAACGCTATGACACCATGAGT...,1
4,ACTGCCCTAAGTGCTCCTTC,AAGAAACTAGGAGAATGGGTAGGCCTTTGTAAAATTGACAGAGAGG...,1



### Use and evaluate the model

In [15]:
# n = 50 is number of samples used as a background image to compare the input with during the shap method
rand_samples = samples.sample(n=50, replace=False, random_state=42).reset_index(drop=True)
background, _ = one_hot_encoding_batch(rand_samples)
deepShap = DeepShap(model, background)

keras is no longer supported, please use tf.keras instead.
Your TensorFlow version is newer than 2.4.0 and so graph support has been removed in eager mode and some static graphs may not be supported. See PR #1483 for discussion.
2025-05-01 18:04:14.201797: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8800
2025-05-01 18:04:14.423604: I tensorflow/stream_executor/cuda/cuda_blas.cc:1786] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


#### #DEBUG Set `threshold_len` to filter out longer genes

In [16]:
threshold_len = -1 # -1
threshold_len_high = -1 # -1

# if threshold_len > 0:
#     count = 0
#     for key,value in gene_symbol_to_seq.items():
#         if(len(value) < threshold_len):
#             count += 1
#     count

if threshold_len > 0:
    count = 0
    for row in sequence_source_df[["ensembl_gene_id", "sequence"]].itertuples():
        if(len(row.sequence) < threshold_len):
            count += 1
    print(count)

In [17]:
# if threshold_len > 0:
#     count_too_long = 0
#     count_kept = 0
#     for key,value in gene_symbol_to_seq.items():
#         if(len(value) > threshold_len):
#             gene_symbol_to_seq[key] = []
#             count_too_long += 1
#         else:
#             count_kept += 1
#     print('count_too_long, len > ', threshold_len, " : ", count_too_long)
#     print('count_kept ', count_kept)


if threshold_len > 0:
    gene_symbol_to_seq = {}
    count_too_long = 0
    count_kept = 0
    for row in sequence_source_df[["ensembl_gene_id", "sequence"]].itertuples():
        # if(len(row.sequence) < threshold_len or len(row.sequence) > threshold_len_high):
            # count_too_long += 1
        # else:
        #     gene_symbol_to_seq[row.ensembl_gene_id] = row.sequence
        #     count_kept += 1
        if(len(row.sequence) < threshold_len): 
            gene_symbol_to_seq[row.ensembl_gene_id] = row.sequence
            count_kept += 1
    print('count_too_long, len > ', threshold_len, " : ", count_too_long)
    print('count_kept ', count_kept)

### Scan the transcript

In [18]:
def score_sequence_attribution_minimal_FIXED(gene, input_miRNA, model, draw_plot=False, step=10, length=50, prediction_threshold=0.5):
    # QUICK FIX - miRBind takes only 20 long miRNA
    miRNA = input_miRNA[0:20]
    miRNAs = []
    genes = []
    counts = np.zeros(len(gene))
    for i in range(0, len(gene) - length + 1, step):
        start = max(i, 0)
        end = min(i+length, len(gene))
        miRNAs.append(miRNA)
        genes.append(gene[start:end])
        counts[start:end] += 1
    labels = np.zeros(len(genes))
    df = pd.DataFrame(
        {'miRNA': miRNAs,
         'gene': genes,
         'label': labels
        })
    data, _ = one_hot_encoding_batch(df, tensor_dim=(50, 20, 1))
    preds = model(data)
    
    attribution = np.zeros((len(gene), len(miRNA)))
    shap_indices = []
    pred_indices = []
    preds_indices = []
    shap_data = []
    counter = 0
    
    for i in range(0, len(gene) - length + 1, step):
        # Access the single prediction value directly
        prediction_score = preds[counter][0]  # Access the first (and only) element
            
        if prediction_score > prediction_threshold:
            shap_indices.append(counter)
            pred_indices.append(i)
            shap_data.append(data[counter])
        counter += 1
    
    # TODO quick fix: if at this point the 'shap_data' is empty array [], it means no prediction was > 0.5, therefore (for now) let's predict 0 binding affinity  
    if len(shap_data) == 0:
        return []
    shap_data = np.stack(shap_data)
    
    neg_shap, pos_shap = deepShap(shap_data)
    
    for i in range(0, len(shap_indices)):
        # Use the single output value directly - no need to index with [1]
        normalized_shap = pos_shap[i,:,:,0] * preds[shap_indices[i]][0]
        newrows = np.zeros((pred_indices[i], normalized_shap.shape[1]))
        normalized_shap = np.vstack([newrows, normalized_shap])
        newrows = np.zeros((len(gene) - pred_indices[i] - length, normalized_shap.shape[1]))
        normalized_shap = np.vstack([normalized_shap, newrows])
        attribution += normalized_shap
    attribution = attribution.T.max(axis=0)
 
    counts[counts == 0] = 1 # because when stepping transcript, its len might not be dividable by step and leave a few 0s at the end of counts
    normalized_scores = attribution / np.array(counts)
    if draw_plot:
        plt.figure(num = random.randint(0, 1000))
        plt.plot(normalized_scores)
        plt.title('normalized_scores => attribution / np.array(counts)')
        plt.show()
    return normalized_scores

In [19]:
def score_with_gradients(gene, input_miRNA, model, draw_plot=False, step=10, length=50, prediction_threshold=0.5):
    # Similar preprocessing as before
    miRNA = input_miRNA[0:20]
    miRNAs = []
    genes = []
    counts = np.zeros(len(gene))
    for i in range(0, len(gene) - length + 1, step):
        start = max(i, 0)
        end = min(i+length, len(gene))
        miRNAs.append(miRNA)
        genes.append(gene[start:end])
        counts[start:end] += 1
        
    labels = np.zeros(len(genes))
    df = pd.DataFrame({
        'miRNA': miRNAs,
        'gene': genes,
        'label': labels
    })
    
    data, _ = one_hot_encoding_batch(df, tensor_dim=(50, 20, 1))
    
    # Get predictions
    preds = model(data)
    
    # Initialize attribution
    attribution = np.zeros(len(gene))
    
    # Use TF's gradient tape for significant predictions
    for idx, i in enumerate(range(0, len(gene) - length + 1, step)):
        if preds[idx][0] > prediction_threshold:
            # Compute gradients for this input
            with tf.GradientTape() as tape:
                x = tf.convert_to_tensor(data[idx:idx+1], dtype=tf.float32)
                tape.watch(x)
                y = model(x)
            
            # Get gradients
            grads = tape.gradient(y, x)[0]
            
            # Sum absolute gradients for feature importance
            # Since we want importance per position in the gene, sum over miRNA dimension
            importance = np.abs(grads.numpy()).sum(axis=2).sum(axis=1).squeeze()
            
            # Map importance back to gene positions
            start = max(i, 0)
            end = min(i+length, len(gene))
            
            # Now we can safely add the importance scores
            # If window size differs from importance length, trim or pad
            window_size = end - start
            if window_size <= len(importance):
                # Use only the needed part of importance
                attribution[start:end] += importance[:window_size] * preds[idx][0].numpy()
            else:
                # Pad with zeros if needed
                padded_importance = np.pad(importance, (0, window_size - len(importance)))
                attribution[start:end] += padded_importance * preds[idx][0].numpy()
    
    # Normalize
    counts[counts == 0] = 1
    normalized_scores = attribution / counts
    
    if draw_plot:
        random_plot_number = random.randint(0, 1000)
        plt.figure(num=random_plot_number)
        plt.plot(normalized_scores)
        plt.title('Gradient-based scores')
        plt.savefig(f'gradient_plots/gradient_scores_plot_{random_plot_number}.png', dpi=300, bbox_inches='tight')  # Save the plot as PNG with good resolution
        plt.show()
    
    return normalized_scores

In [20]:
len(gene_symbol_to_seq.items())

8288

In [21]:
import time

# two objects to collect results for now, will see which one is more handy
score_table = []
miRNA_to_gene_score = {}
explain_errors = []

start = time.time()

i = 0
for miRNA in mirna_sequences[:1]:
    if miRNA not in miRNA_to_gene_score:
        miRNA_to_gene_score[miRNA] = []
    for gene_symbol, gene_sequence in gene_symbol_to_seq.items():
        if not isinstance(gene_sequence, float) and len(gene_sequence) > 0:
            try:
                score = score_with_gradients(
                    str(gene_sequence), 
                    miRNA, 
                    model, 
                    draw_plot=False, 
                    step=10, 
                    length=50, 
                    prediction_threshold=PREDICTION_THRESHOLD
                )
                score_table.append([miRNA, gene_symbol, score])
                miRNA_to_gene_score[miRNA].append([gene_symbol, score])
            except (AssertionError, ValueError) as e:
                # print(miRNA, gene_symbol, e)
                print(e)
                explain_errors.append([mirna_name, miRNA, gene_symbol, str(e)])
                
        else:
            explain_errors.append(gene_symbol)

        i+=1
        # if i % 500 == 0:
        if i % 100 == 0:
            print(gene_symbol, " |",i , "| " , end =" ")
        # if i > 1000:
            # break
    # break
        
            
end = time.time()

NM_000292  | 100 |  NM_000729  | 200 |  NM_001003785  | 300 |  NM_001012456  | 400 |  NM_001029976  | 500 |  NM_001040183  | 600 |  NM_001077692  | 700 |  NM_001098832  | 800 |  NM_001122834  | 900 |  NM_001130067  | 1000 |  NM_001135664  | 1100 |  NM_001142778  | 1200 |  NM_001152  | 1300 |  NM_201277  | 8200 |  

In [22]:
print(f'{round(end - start, 2)} seconds, {round((end - start) / 3600, 2)} hours')

19073.62 seconds, 5.3 hours


In [23]:
len(gene_symbol_to_seq.items())

8288

In [24]:
explain_errors

[]

In [25]:
empties = 0
for gene_n_score in miRNA_to_gene_score[my_miRNA]:
    if len(gene_n_score[1]) == 0:
        empties += 1
print(empties, ' / ', len(miRNA_to_gene_score[my_miRNA]))

0  /  8288


#### Save the explainability scoring to a file

In [26]:
print(type(miRNA_to_gene_score), type(miRNA_to_gene_score[my_miRNA]))

<class 'dict'> <class 'list'>


In [27]:
SAVE_EXPLAINABILITY_SCORES_PATH

'data/scanned/GRCh37.p13 hg19/UCSC/3utr.sequences.refseq_id.mirna_fc.gradient_scores_hsa-miR-103a-3p.json'

In [28]:
if SAVE_EXPL_SCORES:
    with open(SAVE_EXPLAINABILITY_SCORES_PATH, 'w') as file:
        data_to_save = {}
        for key in miRNA_to_gene_score.keys():
            data_to_save[key] = [[sub_key, list(sub_val)] for sub_key, sub_val in miRNA_to_gene_score[key]]

        json.dump(data_to_save, file, indent=4)

In [29]:
data_to_save.keys()

dict_keys(['AGCAGCATTGTACAGGGCTATGA'])

In [30]:
if SAVE_EXPL_SCORES:
    with open(SAVE_SCANNING_ERRORS_PATH, 'w') as filehandle:
        json.dump(explain_errors, filehandle)

In [33]:
SAVE_SCANNING_ERRORS_PATH

'data/scanned/GRCh37.p13 hg19/UCSC/3utr.sequences.refseq_id.mirna_fc.gradient_scores_hsa-miR-103a-3p.scanning_errors.txt'

In [32]:
# plt.plot(miRNA_to_gene_score['TGAGGTAGTAGGTTGTATGGTT'][0][1])